# Импорт зависимостей

In [1]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.downloader import download_warc_files
from src.data.converter import convert_all_warc
from src.data.cleaner import clean_all_jsonl
from src.data.entropy import compute_dataset_entropy_dir, filter_dataset_dir
import src.tokenization.tokenizers as tk
import torch

from pathlib import Path
import random
import json

from datasets import load_dataset
import json
from pathlib import Path

import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset

from pathlib import Path

import json
from pathlib import Path

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

W0830 23:50:08.522000 15628 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


CUDA available: True
NVIDIA GeForce RTX 2060 SUPER


# 3.1 Скачать датасет

## 1. Скачать часть Comon Crawl - WARC файлы

In [2]:
download_warc_files(config_path="../configs/config.yaml", raw_data_dir="../data/raw/common_crawl/warc")

23:50:08 | INFO     | Директория для сохранения: R:\Folders\1. master_degree_ITMO\semester 2\LABS\MNNA-2026\MNNA-2026-labs-DmitrievDM\data\raw\common_crawl\warc


Общий прогресс:   0%|          | 0/1 [00:00<?, ?it/s]

23:50:08 | INFO     | Начинаем скачивание: CC-MAIN-20260605214811-20260606004811-00000.warc.gz


CC-MAIN-20260605214811-20260606004811-00000.warc.gz:   0%|          | 0.00/940M [00:00<?, ?B/s]

23:51:38 | INFO     | ✅ Успешно скачано: CC-MAIN-20260605214811-20260606004811-00000.warc.gz


## 2. Конверитровать WARC файлы в текстовый формат.

In [3]:
convert_all_warc(input_dir="../data/raw/common_crawl/warc", output_dir="../data/raw/common_crawl/jsonl")

23:51:38 | INFO     | Найдено WARC-файлов: 1


Конвертация WARC:   0%|          | 0/1 [00:00<?, ?file/s]

23:51:38 | INFO     | Обработка файла: ..\data\raw\common_crawl\warc\CC-MAIN-20260605214811-20260606004811-00000.warc.gz
23:52:43 | INFO     | Файл ..\data\raw\common_crawl\warc\CC-MAIN-20260605214811-20260606004811-00000.warc.gz обработан: всего записей=63412, записано=20820, ошибок=0
23:52:43 | INFO     | Готово: ..\data\raw\common_crawl\jsonl\CC-MAIN-20260605214811-20260606004811-00000.jsonl
23:52:43 | INFO     | Обработка завершена
23:52:43 | INFO     | Файлов найдено: 1
23:52:43 | INFO     | Сконвертировано: 1
23:52:43 | INFO     | Пропущено: 0
23:52:43 | INFO     | Ошибок: 0


# 3.2 Очистка данных

In [4]:
clean_all_jsonl(input_dir="../data/raw/common_crawl/jsonl", output_dir="../data/processed/common_crawl/stage1_cleaned")

23:52:43 | INFO     | Обработка: ..\data\raw\common_crawl\jsonl\CC-MAIN-20260605214811-20260606004811-00000.jsonl
00:11:53 | INFO     | Файл ..\data\raw\common_crawl\jsonl\CC-MAIN-20260605214811-20260606004811-00000.jsonl обработан: прочитано=20820, оставлено объектов=10012, отброшено=10808, итоговых чанков записано=24057


***Примечания:*** *стоит отладить код, добавить прогресс-бар, посмотреть как работает фильтрация и нужна ли она, также выяснить причины очень долгого выполнения очистки данных*

# 3.3 Улучшить качество данных

## 1. GPT2 и энтропия

In [5]:
stats = compute_dataset_entropy_dir(
    input_dir="../data/processed/common_crawl/stage1_cleaned",
    output_dir="../data/metrics/common_crawl/entropy_stats",
    stats_path="../data/metrics/common_crawl/full_entropy_stats.json",
    pattern="*.jsonl",
    model_name="gpt2",
    text_field="text",
    batch_size=8,
    max_length=1024,
)

print(stats["info_density_nats_per_token"])

00:11:53 | INFO     | Using device: cuda
00:11:53 | INFO     | HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
00:11:53 | WARNING  | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
00:11:54 | INFO     | HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
00:11:54 | INFO     | HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
00:11:54 | INFO     | HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
00:11:54 | INFO     | HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
00:11:54 | INFO     | HTTP Request: GET https://hu

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

00:11:57 | INFO     | HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/generation_config.json "HTTP/1.1 200 OK"
00:11:59 | INFO     | Processing file: ..\data\processed\common_crawl\stage1_cleaned\CC-MAIN-20260605214811-20260606004811-00000.jsonl


CC-MAIN-20260605214811-20260606004811-00000.jsonl: 0batch [00:00, ?batch/s]

01:03:19 | INFO     | File CC-MAIN-20260605214811-20260606004811-00000.jsonl done. Objects: 24057, info density: 2.798229
01:03:19 | INFO     | Global info density: 2.798229


2.7982289753131053


## 2. Удаление дубликатов и объектов с высокой или низкой энтропией

In [6]:
filter_dataset_dir(
    original_dir="../data/processed/common_crawl/stage1_cleaned",               # Папка с ИСХОДНЫМ датасетом
    metrics_dir="../data/metrics/common_crawl/entropy_stats",    # Папка, куда предыдущий код сохранил энтропию
    output_dir="../data/processed/common_crawl/stage2_filtered",       # Папка, куда сохранится ИТОГОВЫЙ чистый датасет
    lower_percentile=1.0,                  # Удалить 1% текстов с самой НИЗКОЙ энтропией
    upper_percentile=99.0                  # Удалить 1% текстов с самой ВЫСОКОЙ энтропией
)

01:03:19 | INFO     | Загрузка метрик из CC-MAIN-20260605214811-20260606004811-00000.jsonl...


01:03:19 | INFO     | Установлены границы энтропии: [0.4008, 6.2341]
01:03:21 | INFO     | Файл CC-MAIN-20260605214811-20260606004811-00000.jsonl обработан.
01:03:21 | INFO     | Статистика: {'total_valid_objects': 24057, 'kept': 22953, 'removed_duplicates': 635, 'removed_low_entropy': 240, 'removed_high_entropy': 229, 'skipped_empty_or_invalid': 0}
01:03:21 | INFO     | ========================================
01:03:21 | INFO     | ГЛОБАЛЬНАЯ СТАТИСТИКА ОЧИСТКИ:
01:03:21 | INFO     | total_valid_objects: 24057
01:03:21 | INFO     | kept: 22953
01:03:21 | INFO     | removed_duplicates: 635
01:03:21 | INFO     | removed_low_entropy: 240
01:03:21 | INFO     | removed_high_entropy: 229
01:03:21 | INFO     | skipped_empty_or_invalid: 0


{'total_valid_objects': 24057,
 'kept': 22953,
 'removed_duplicates': 635,
 'removed_low_entropy': 240,
 'removed_high_entropy': 229,
 'skipped_empty_or_invalid': 0}

# 3.4 Токенизация

In [ ]:

# =========================
# 1. Настройки
# =========================
# Папка, куда filter_dataset_dir сохранил очищенные .jsonl файлы
FILTERED_DATA_DIR = Path("../data/processed/common_crawl/stage2_filtered") 
TEXT_FIELD = "text"
SEED = 42
OUTPUT_DIR = Path("../data/tokenizers")

MAX_TEXTS = 200_000 

# =========================
# 2. Простой цикл для чтения всех .jsonl файлов
# =========================
print(f"Поиск .jsonl файлов в {FILTERED_DATA_DIR}...")
jsonl_files = sorted(FILTERED_DATA_DIR.glob("*.jsonl"))

if not jsonl_files:
    raise FileNotFoundError(f"Не найдено .jsonl файлов в {FILTERED_DATA_DIR}")

print(f"Найдено файлов: {len(jsonl_files)}")
print("Чтение текстов...")

texts = []
for file_path in jsonl_files:
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            try:
                obj = json.loads(line)
                text = obj.get(TEXT_FIELD, "")
                if text:
                    texts.append(str(text))
            except json.JSONDecodeError:
                continue
                
            # Прерываем чтение, если достигли лимита
            if MAX_TEXTS is not None and len(texts) >= MAX_TEXTS:
                break
                
    if MAX_TEXTS is not None and len(texts) >= MAX_TEXTS:
        print(f"Достигнут лимит в {MAX_TEXTS} текстов. Останавливаем чтение.")
        break

print(f"Итого загружено текстов: {len(texts)}")
assert len(texts) > 0, "Список текстов пуст!"

# Выбираем случайный объект для демонстрации
rng = random.Random(SEED)
sample_idx = rng.randrange(len(texts))
sample_text = texts[sample_idx]

print("-" * 60)
print(f"Индекс случайного объекта: {sample_idx}")
print(f"Пример текста (первые 150 символов): {sample_text[:150]}")

Поиск .jsonl файлов в ..\data\processed\common_crawl\stage2_filtered...
Найдено файлов: 1
Чтение текстов...
Итого загружено текстов: 22953
------------------------------------------------------------
Индекс случайного объекта: 20952
Пример текста (первые 150 символов): In wood building traditions and housing stock structures, Finland and Sweden are more similar (i.e., traditions in wood building, larger proportion of


## 3.4.1 Символьная токенизация

In [8]:
print("-" * 60)
print("Обучение символьного токенизатора...")
char_token2id = tk.fit_char_tokenizer(texts, max_samples=None, seed=SEED)
char_ids = tk.encode_char(sample_text, char_token2id, add_special=True)

print("1. Токенизация по символам")
print(f"Размер словаря: {len(char_token2id)}")
print(f"Размер последовательности случайного объекта: {len(char_ids)}")
# Сохранение
char_path = tk.save_vocab(char_token2id, OUTPUT_DIR / "char_tokenizer.json")

------------------------------------------------------------
Обучение символьного токенизатора...
1. Токенизация по символам
Размер словаря: 166
Размер последовательности случайного объекта: 5133


## 3.4.2 Токенизация по словам

In [9]:
print("-" * 60)
print("Обучение словесного токенизатора...")
word_token2id = tk.fit_word_tokenizer(
    texts,
    max_samples=100_000, 
    max_vocab_size=30_000,
    min_freq=2,
    seed=SEED
)
word_ids = tk.encode_word(sample_text, word_token2id, add_special=True)

print("2. Токенизация по словам")
print(f"Размер словаря: {len(word_token2id)}")
print(f"Размер последовательности случайного объекта: {len(word_ids)}")
# Сохранение
word_path = tk.save_vocab(word_token2id, OUTPUT_DIR / "word_tokenizer.json")

------------------------------------------------------------
Обучение словесного токенизатора...
2. Токенизация по словам
Размер словаря: 30004
Размер последовательности случайного объекта: 784


## 3.4.3 BPE

In [10]:
print("-" * 60)
print("Обучение BPE токенизатора (может занять пару минут)...")
bpe_tokenizer = tk.train_bpe_tokenizer(
    texts,
    max_samples=100_000,
    vocab_size=10_000,
    min_freq=2,
    seed=SEED
)
bpe_ids = tk.encode_bpe(sample_text, bpe_tokenizer, add_special=True)

print("3. BPE-токенизация")
print(f"Размер словаря: {bpe_tokenizer.get_vocab_size()}")
print(f"Размер последовательности случайного объекта: {len(bpe_ids)}")
# Сохранение
bpe_path = tk.save_bpe_tokenizer(bpe_tokenizer, OUTPUT_DIR / "bpe_tokenizer.json")

------------------------------------------------------------
Обучение BPE токенизатора (может занять пару минут)...
3. BPE-токенизация
Размер словаря: 10000
Размер последовательности случайного объекта: 1179


# 3.5 Обучение на wikitext

## 3.5.1 Скачивание датасета Wikitext

In [11]:


raw_wiki_dir = Path("../data/raw/wikitext")
raw_wiki_dir.mkdir(parents=True, exist_ok=True)

# Использовать можно либо: wikitext-103-raw-v1 либо если долго: wikitext-2-raw-v1.
print("Скачивание wikitext...")
wiki = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

for split in ["train", "validation", "test"]:
    out_path = raw_wiki_dir / f"{split}.jsonl"
    print(f"Сохранение {split} в {out_path}...")
    with out_path.open("w", encoding="utf-8") as f:
        for item in wiki[split]:
            text = item["text"].strip()
            if text:
                f.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")
                
print("Скачивание и конвертация в JSONL завершены.")

Скачивание wikitext...


01:03:34 | INFO     | HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
01:03:34 | INFO     | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Salesforce/wikitext/b08601e04326c79dfdd32d625aee71d232d685c3/README.md "HTTP/1.1 200 OK"
01:03:34 | INFO     | HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 404 Not Found"
01:03:35 | INFO     | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Salesforce/wikitext/Salesforce/wikitext.py "HTTP/1.1 404 Not Found"
01:03:35 | INFO     | HTTP Request: GET https://huggingface.co/api/datasets/Salesforce/wikitext/revision/b08601e04326c79dfdd32d625aee71d232d685c3 "HTTP/1.1 200 OK"
01:03:35 | INFO     | HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/.huggingf

Сохранение train в ..\data\raw\wikitext\train.jsonl...
Сохранение validation в ..\data\raw\wikitext\validation.jsonl...
Сохранение test в ..\data\raw\wikitext\test.jsonl...
Скачивание и конвертация в JSONL завершены.


## 3.5.2 Очистка, Энтропия и Фильтрация

In [15]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

print("1. Очистка данных wikitext...")
clean_all_jsonl(input_dir="../data/raw/wikitext", output_dir="../data/processed/wikitext/stage1_cleaned", text_field="text", use_toxic_filter=False, lang_detect=False)

raw_dir = Path("../data/raw/wikitext")
cleaned_dir = Path("../data/processed/wikitext/stage1_cleaned")

cleaned_dir.mkdir(parents=True, exist_ok=True)

# Проверка
print("\nПроверка количества строк в cleaned_wikitext:")
for f in sorted(cleaned_dir.glob("*.jsonl")):
    lines = sum(1 for _ in f.open("r", encoding="utf-8"))
    print(f"  {f.name}: {lines} строк")
# ===========================================================
print("\n2. Вычисление энтропии (GPT-2)...")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

stats = compute_dataset_entropy_dir(
    input_dir="../data/processed/wikitext/stage1_cleaned",
    output_dir="../data/metrics/wikitext/entropy_stats",
    stats_path="../data/metrics/wikitext/full_entropy_stats.json",
    pattern="*.jsonl",
    model_name="gpt2",
    text_field="text",
    batch_size=32,
    max_length=512,
)
print(f"Информационная плотность wikitext: {stats['info_density_nats_per_token']:.4f}")

print("\n3. Фильтрация дубликатов и экстремальной энтропии...")
filter_dataset_dir(
    original_dir="../data/processed/wikitext/stage1_cleaned",
    metrics_dir="../data/metrics/wikitext/entropy_stats",
    output_dir="../data/processed/wikitext/stage2_filtered",
    lower_percentile=1.0,
    upper_percentile=99.0
)
print("\nПайплайн подготовки wikitext завершен!")

08:30:45 | INFO     | Обработка: ..\data\raw\wikitext\test.jsonl


CUDA available: True
NVIDIA GeForce RTX 2060 SUPER
1. Очистка данных wikitext...


08:30:47 | INFO     | Файл ..\data\raw\wikitext\test.jsonl обработан: прочитано=2891, оставлено объектов=2574, отброшено=317, итоговых чанков записано=2574
08:30:47 | INFO     | Обработка: ..\data\raw\wikitext\train.jsonl
08:38:20 | INFO     | Файл ..\data\raw\wikitext\train.jsonl обработан: прочитано=1165029, оставлено объектов=1042440, отброшено=122589, итоговых чанков записано=1042445
08:38:20 | INFO     | Обработка: ..\data\raw\wikitext\validation.jsonl
08:38:21 | INFO     | Файл ..\data\raw\wikitext\validation.jsonl обработан: прочитано=2461, оставлено объектов=2226, отброшено=235, итоговых чанков записано=2226



Проверка количества строк в cleaned_wikitext:
  test.jsonl: 2574 строк


08:38:24 | INFO     | Using device: cuda


...

validation.jsonl: 0batch [00:00, ?batch/s]

15:00:50 | INFO     | File validation.jsonl done. Objects: 2226, info density: 3.894513
15:00:50 | INFO     | Global info density: 3.880776
15:00:50 | INFO     | Загрузка метрик из test.jsonl...
15:00:50 | INFO     | Установлены границы энтропии: [2.9286, 8.4260]


Информационная плотность wikitext: 3.8808

3. Фильтрация дубликатов и экстремальной энтропии...


15:00:51 | INFO     | Файл test.jsonl обработан.
15:00:51 | INFO     | Статистика: {'total_valid_objects': 2574, 'kept': 2492, 'removed_duplicates': 30, 'removed_low_entropy': 26, 'removed_high_entropy': 26, 'skipped_empty_or_invalid': 0}
15:00:51 | INFO     | Загрузка метрик из train.jsonl...
15:00:57 | INFO     | Установлены границы энтропии: [2.9054, 8.2000]
15:01:23 | INFO     | Файл train.jsonl обработан.
15:01:23 | INFO     | Статистика: {'total_valid_objects': 1042445, 'kept': 923483, 'removed_duplicates': 99766, 'removed_low_entropy': 10263, 'removed_high_entropy': 8933, 'skipped_empty_or_invalid': 0}
15:01:24 | INFO     | Загрузка метрик из validation.jsonl...
15:01:24 | INFO     | Установлены границы энтропии: [2.9523, 8.3960]
15:01:24 | INFO     | Файл validation.jsonl обработан.
15:01:24 | INFO     | Статистика: {'total_valid_objects': 2226, 'kept': 2163, 'removed_duplicates': 17, 'removed_low_entropy': 23, 'removed_high_entropy': 23, 'skipped_empty_or_invalid': 0}
15:01:24


Пайплайн подготовки wikitext завершен!


## 3.5.3 Реализация Packed Batching и LightningDataModule

In [16]:


def create_packed_batches(texts, tokenizer, max_length=512):
    """
    Packed batching с локальной маской:
    0 — PAD, 1 — первый объект, 2 — второй объект и т.д.
    """
    packs = []

    current_input_ids = []
    current_mask = []
    segment_id = 1  # локальный ID внутри текущего пака

    pad_id = tokenizer.token_to_id("<pad>")
    eos_id = tokenizer.token_to_id("<eos>")

    for text in texts:
        ids = tokenizer.encode(text).ids

        if eos_id is not None:
            ids.append(eos_id)

        current_input_ids.extend(ids)
        current_mask.extend([segment_id] * len(ids))
        segment_id += 1

        # Набиваем полные паки
        while len(current_input_ids) >= max_length:
            pack_input_ids = current_input_ids[:max_length]
            pack_mask = current_mask[:max_length]

            # Нормализуем маску: 1, 2, 3...
            # (на случай, если в пак попали куски одного и того же документа)
            unique_ids = sorted(set(pack_mask))
            id_map = {old: new for new, old in enumerate(unique_ids, start=1)}
            pack_mask = [id_map[m] for m in pack_mask]

            packs.append({
                "input_ids": pack_input_ids,
                "mask": pack_mask
            })

            # Остаток переносим в новый пак
            remaining_ids = current_input_ids[max_length:]
            remaining_mask = current_mask[max_length:]

            # Сбрасываем segment_id для нового пака
            # Остаток принадлежит тому же документу, что и конец предыдущего пака
            current_input_ids = remaining_ids
            current_mask = remaining_mask
            segment_id = max(id_map.values()) + 1 if remaining_mask else 1

    # Последний неполный пак — паддим до max_length
    if len(current_input_ids) > 0:
        pad_len = max_length - len(current_input_ids)

        # Нормализуем маску перед паддингом
        unique_ids = sorted(set(current_mask))
        id_map = {old: new for new, old in enumerate(unique_ids, start=1)}
        current_mask = [id_map[m] for m in current_mask]

        pack_input_ids = current_input_ids + [pad_id] * pad_len
        pack_mask = current_mask + [0] * pad_len  # 0 для PAD

        packs.append({
            "input_ids": pack_input_ids,
            "mask": pack_mask
        })

    return packs


class PackedWikiTextDataset(Dataset):
    def __init__(self, packs):
        self.packs = packs
        
    def __len__(self):
        return len(self.packs)
        
    def __getitem__(self, idx):
        pack = self.packs[idx]
        return {
            "input_ids": torch.tensor(pack["input_ids"], dtype=torch.long),
            "mask": torch.tensor(pack["mask"], dtype=torch.long),
        }


class WikiTextDataModule(pl.LightningDataModule):
    def __init__(self, data_dir, tokenizer_path, max_length=512, batch_size=4):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.tokenizer_path = tokenizer_path
        self.max_length = max_length
        self.batch_size = batch_size
        
    def setup(self, stage=None):
        # Загружаем BPE, обученный на Common Crawl!
        self.tokenizer = tk.load_bpe_tokenizer(self.tokenizer_path)
        
        self.train_packs = self._pack_split("train.jsonl")
        self.val_packs = self._pack_split("validation.jsonl")
        
    def _pack_split(self, filename):
        texts = []
        file_path = self.data_dir / filename
        if not file_path.exists():
            return []
            
        with file_path.open("r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                if obj.get("text"):
                    texts.append(obj["text"])
                    
        return create_packed_batches(texts, self.tokenizer, self.max_length)
        
    def train_dataloader(self):
        dataset = PackedWikiTextDataset(self.train_packs)
        return DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        
    def val_dataloader(self):
        dataset = PackedWikiTextDataset(self.val_packs)
        return DataLoader(dataset, batch_size=self.batch_size)

## 3.5.4 Демонстрация

In [17]:
# Инициализируем DataModule

dm = WikiTextDataModule(
    data_dir="../data/processed/wikitext/stage2_filtered",
    tokenizer_path="../data/tokenizers/bpe_tokenizer.json", # BPE от Common Crawl
    max_length=512,
    batch_size=2
)

dm.setup()

# Берем один батч из train_dataloader
train_loader = dm.train_dataloader()
batch = next(iter(train_loader))

print("Размер батча (input_ids):", batch["input_ids"].shape)
print("Размер батча (mask):", batch["mask"].shape)

# Посмотрим на первый объект в батче
print("\n--- Первый объект в батче ---")
print("Первые 20 input_ids:", batch["input_ids"][0][:20].tolist(), "...")
print("Первые 20 mask:", batch["mask"][0][:20].tolist(), "...")

# Проверим, как работает маска в конце последовательности (где должны быть PAD токены)
print("\nМаска в конце первого объекта (должны быть нули для PAD):")
print("mask:", batch["mask"][0][-20:].tolist())

# Проверим, есть ли склейка разных объектов (значения маски 1, 2, 3...)
unique_segments = torch.unique(batch["mask"][0]).tolist()
print(f"\nУникальные ID объектов в первом батче (0 - это PAD): {unique_segments}")
print("Если в списке есть числа больше 1 (например, 1, 2, 3), значит packed batching успешно склеил несколько текстов в один!")

Размер батча (input_ids): torch.Size([2, 512])
Размер батча (mask): torch.Size([2, 512])

--- Первый объект в батче ---
Первые 20 input_ids: [291, 526, 1074, 185, 8509, 669, 65, 4517, 5970, 197, 893, 4173, 320, 4205, 11, 6239, 5826, 18, 72, 12] ...
Первые 20 mask: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2] ...

Маска в конце первого объекта (должны быть нули для PAD):
mask: [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]

Уникальные ID объектов в первом батче (0 - это PAD): [1, 2, 3, 4]
Если в списке есть числа больше 1 (например, 1, 2, 3), значит packed batching успешно склеил несколько текстов в один!


In [18]:
# Берём самый последний пак из train
last_pack = dm.train_packs[-1]

print("Маска в конце последнего пака (должны быть нули - PAD):")
print(last_pack["mask"][-30:])

unique_segments = sorted(list(set(last_pack["mask"])))
print(f"\nУникальные ID в последнем паке (0 - это PAD): {unique_segments}")

Маска в конце последнего пака (должны быть нули - PAD):
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Уникальные ID в последнем паке (0 - это PAD): [0, 1, 2]
